# Fase 4 — Modelo Gold (estrella)

Silver → dimensiones + hechos + vistas KPI + `dim_paciente_bi` (sin PII).

Prerrequisitos: Bronze y Silver completos.

In [ ]:
dbutils.widgets.text("repo_root", "", "Ruta Repo Databricks")
dbutils.widgets.text("batch_id", "", "Batch ID (vacío = desde silver)")
dbutils.widgets.dropdown("optimize_delta", "true", ["true", "false"], "OPTIMIZE ZORDER")

In [ ]:
import sys
from uuid import uuid4

repo_root = dbutils.widgets.get("repo_root").rstrip("/")
if repo_root:
    sys.path.insert(0, f"{repo_root}/src")

from ips_analytics.config import DEFAULT_CONFIG
from ips_analytics.gold.build_star_schema import run_gold_pipeline

catalog = DEFAULT_CONFIG.catalog
batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    batch_id = (
        spark.table(f"{catalog}.silver.pacientes")
        .select("_batch_id")
        .limit(1)
        .collect()[0]["_batch_id"]
    )

do_opt = dbutils.widgets.get("optimize_delta") == "true"
result = run_gold_pipeline(
    spark,
    batch_id=batch_id,
    run_id=str(uuid4()),
    optimize=do_opt,
)
for k, v in sorted(result.row_counts.items()):
    print(f"{k}: {v}")

In [ ]:
silver_citas = spark.table(f"{catalog}.silver.citas").count()
gold_citas = spark.table(f"{catalog}.gold.fact_citas").count()
print(f"silver.citas={silver_citas} fact_citas={gold_citas}")
assert silver_citas == gold_citas, "Q-G01: conteo citas Silver vs Gold"

from pyspark.sql import functions as F
s_neto = spark.table(f"{catalog}.silver.facturacion").agg(F.sum("valor_neto")).collect()[0][0]
g_neto = spark.table(f"{catalog}.gold.fact_facturacion").agg(F.sum("valor_neto")).collect()[0][0]
print(f"sum valor_neto silver={s_neto} gold={g_neto}")
assert abs(float(s_neto) - float(g_neto)) < 0.01, "Q-G02: reconciliación facturación"
print("Validación Gold OK")

In [ ]:
spark.table(f"{catalog}.gold.v_kpi_resumen_ips").orderBy("anio", "mes").show(12)

## Cierre Fase 4

Power BI: conectar a `gold.fact_*`, vistas `v_kpi_*` y **`dim_paciente_bi`** (no usar PII en claro).

Siguiente: **Fase 5** (incremental ops) o **Fase 6** (dashboard).